In [ ]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では無視され、Colab環境でのみ自動でモジュールをインストールします
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/study-deep-learning'):
        print("リポジトリをダウンロード中...")
        !git clone https://github.com/miyayudai/study-deep-learning.git > /dev/null 2>&1
    
    print("必要なモジュールをインストール中...")
    %cd /content/study-deep-learning
    !pip install -q -r requirements.txt
    !pip install -q -e .
    
    print("作業ディレクトリをセットアップ中...")
    %cd /content/study-deep-learning/4
    print("準備完了！このまま下のセルを実行できます。")

# 第4章 単層ネットワーク: 回帰 (Single-layer Networks: Regression)

## 演習問題 (Exercises 4.1 〜 4.12)

---

### 本ノートブックの構成と目的
本ノートブックでは、Bishop & Bishop (2024) *Deep Learning: Foundations and Concepts* 第4章末に収録された**演習問題 4.1 〜 4.12 (全12問)** を完全に解答・解説します。

各問題において、以下の要素を網羅しています：
1. **問題文（英語原文および日本語訳）**
2. **完全な数学的証明と数式展開（LaTeX）**
3. **Python による数値的検証・シミュレーションコード**
4. **自己採点アサーション (`assert`) による完全性の保証**

---

### 問題一覧
- **Exercise 4.1 (?)**: 多項式曲線あてはめにおける二乗和誤差最小化と連立一次方程式（正規方程式）の導出
- **Exercise 4.2 (?)**: 正則化項を含む多項式曲線あてはめの連立線形方程式の導出
- **Exercise 4.3 (?)**: $\tanh$ 関数とロジスティック・シグモイド関数の等価性とネットワークパラメータの相互変換
- **Exercise 4.4 (? ? ?)**: 射影行列 $\mathbf{P} = \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T$ の代数的性質と最小二乗法の直交射影幾何学の証明
- **Exercise 4.5 (?)**: 重み付き最小二乗法 (WLS) の最適解と、不均一ノイズ分散・データ重複の2つの統計的解釈
- **Exercise 4.6 (?)**: 正則化二乗和誤差関数の勾配計算による厳密な最小値の導出
- **Exercise 4.7 (? ?)**: 一般共分散行列 $\boldsymbol{\Sigma}$ を持つ多変量線形回帰モデルの最尤推定（$\mathbf{W}_{\mathrm{ML}}$ が $\boldsymbol{\Sigma}$ に不変である証明と $\boldsymbol{\Sigma}_{\mathrm{ML}}$ の導出）
- **Exercise 4.8 (?)**: 変分法による多変量二乗損失の期待値最小化（最適予測が条件付き期待値である証明）
- **Exercise 4.9 (?)**: 多変量二乗損失の二乗展開と交差項消失による直交分解の証明
- **Exercise 4.10 (? ?)**: 展開結果から条件付き期待値の最適性を直接再導出
- **Exercise 4.11 (? ?)**: 一般化ガウス分布の規格化証明（ガンマ関数）、$q=2$ でのガウスへの帰着、対数尤度と $L_q$ 損失最小化の等価性
- **Exercise 4.12 (? ?)**: $L_q$ 損失の停留条件、$q=1$ での条件付き中央値、$q \to 0$ での条件付き最頻値の厳密な数学的導出


In [1]:
# 必要なライブラリのインポートと共通設定
import os
import sys
import numpy as np
import scipy.linalg as la
import scipy.special as sp
import scipy.integrate as integrate
import scipy.optimize as opt
import matplotlib.pyplot as plt

repo_root = os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from common.exercises_ch4 import (
    solve_polynomial_normal_equations,
    solve_regularized_polynomial_equations,
    sigmoid_to_tanh_params,
    eval_sigmoid_network,
    eval_tanh_network,
    compute_projection_matrix,
    solve_weighted_least_squares,
    estimate_multivariate_regression_mle,
    generalized_gaussian_pdf,
    generalized_gaussian_log_likelihood,
    compute_optimal_lq_predictor
)
from common.plot_utils import setup_style

setup_style()
print("Setup completed successfully!")


Setup completed successfully!


---
## Exercise 4.1 (?): 多項式曲線あてはめの正規方程式

### 問題文
Consider the sum-of-squares error function given by (1.2) in which the function $y(x, \mathbf{w})$ is given by the polynomial (1.1). Show that the coefficients $\mathbf{w} = \{w_i\}$ that minimize this error function are given by the solution to the following set of linear equations:
$$
\sum_{j=0}^M A_{ij} w_j = T_i \tag{4.53}
$$
where
$$
A_{ij} = \sum_{n=1}^N (x_n)^{i+j}, \quad T_i = \sum_{n=1}^N (x_n)^i t_n \tag{4.54}
$$
Here a suffix $i$ or $j$ denotes the index of a component, whereas $(x)^i$ denotes $x$ raised to the power of $i$.

---

### 数学的証明
多項式モデル $y(x, \mathbf{w})$ は次の形をしています：
$$
y(x, \mathbf{w}) = \sum_{j=0}^M w_j x^j
$$
二乗和誤差関数 $E_D(\mathbf{w})$ は式 (1.2) より：
$$
E_D(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \{y(x_n, \mathbf{w}) - t_n\}^2 = \frac{1}{2} \sum_{n=1}^N \left\{ \sum_{j=0}^M w_j x_n^j - t_n \right\}^2
$$
この誤差関数を特定の係数 $w_i$ ($i = 0, \dots, M$) に関して偏微分します：
$$
\frac{\partial E_D}{\partial w_i} = \sum_{n=1}^N \left\{ \sum_{j=0}^M w_j x_n^j - t_n \right\} \frac{\partial y(x_n, \mathbf{w})}{\partial w_i}
$$
ここで $\frac{\partial y(x_n, \mathbf{w})}{\partial w_i} = x_n^i$ であるため：
$$
\frac{\partial E_D}{\partial w_i} = \sum_{n=1}^N \left\{ \sum_{j=0}^M w_j x_n^j - t_n \right\} x_n^i = \sum_{j=0}^M w_j \left( \sum_{n=1}^N x_n^{i+j} \right) - \sum_{n=1}^N x_n^i t_n
$$
誤差関数が最小となる停留点では、すべての $i = 0, \dots, M$ について $\frac{\partial E_D}{\partial w_i} = 0$ が成り立ちます：
$$
\sum_{j=0}^M \left( \sum_{n=1}^N x_n^{i+j} \right) w_j = \sum_{n=1}^N x_n^i t_n
$$
ここで $A_{ij} = \sum_{n=1}^N (x_n)^{i+j}$、および $T_i = \sum_{n=1}^N (x_n)^i t_n$ と定義すると、連立一次方程式：
$$
\sum_{j=0}^M A_{ij} w_j = T_i \quad (i = 0, \dots, M)
$$
が得られます。行列形式では $\mathbf{A} \mathbf{w} = \mathbf{T}$（ここで $\mathbf{A} = \boldsymbol{\Phi}^T \boldsymbol{\Phi}, \mathbf{T} = \boldsymbol{\Phi}^T \mathbf{t}$）となり、題意が示されました。 $\blacksquare$


In [2]:
# Exercise 4.1 の数値検証
rng = np.random.RandomState(42)
x_sample = np.linspace(-1, 1, 25)
t_sample = 1.5 * x_sample**2 - 2.0 * x_sample + 0.8 + rng.normal(0, 0.05, len(x_sample))
M = 2

# 式 (4.53)-(4.54) に基づく解
w_ex1, A_ex1, T_ex1 = solve_polynomial_normal_equations(x_sample, t_sample, M)

# Vandermonde 行列による直接解 Phi w = t
Phi_ex1 = np.vander(x_sample, M + 1, increasing=True)
w_direct = la.solve(Phi_ex1.T @ Phi_ex1, Phi_ex1.T @ t_sample, assume_a='pos')

print("Exercise 4.1 解 w:", w_ex1)
print("直接解 w_direct:  ", w_direct)
assert np.allclose(w_ex1, w_direct, atol=1e-12)
print("Exercise 4.1: 検証成功！")


Exercise 4.1 解 w: [ 0.78180097 -2.0320686   1.52775774]
直接解 w_direct:   [ 0.78180097 -2.0320686   1.52775774]
Exercise 4.1: 検証成功！


---
## Exercise 4.2 (?): 正則化多項式曲線あてはめの連立線形方程式

### 問題文
Write down the set of coupled linear equations, analogous to (4.53), satisfied by the coefficients $w_i$ that minimize the regularized sum-of-squares error function given by (1.4).

---

### 数学的証明
正則化二乗和誤差関数 (式 1.4) は次式で与えられます：
$$
E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \{y(x_n, \mathbf{w}) - t_n\}^2 + \frac{\lambda}{2} \|\mathbf{w}\|^2 = \frac{1}{2} \sum_{n=1}^N \left\{ \sum_{j=0}^M w_j x_n^j - t_n \right\}^2 + \frac{\lambda}{2} \sum_{j=0}^M w_j^2
$$
$w_i$ に関する偏微分を計算します：
$$
\frac{\partial E}{\partial w_i} = \frac{\partial E_D}{\partial w_i} + \lambda w_i = \sum_{j=0}^M A_{ij} w_j - T_i + \lambda w_i
$$
クロネッカーのデルタ $\delta_{ij}$ を用いると、$w_i = \sum_{j=0}^M \delta_{ij} w_j$ と書けるため：
$$
\frac{\partial E}{\partial w_i} = \sum_{j=0}^M (A_{ij} + \lambda \delta_{ij}) w_j - T_i
$$
極小値を与える停留点 $\frac{\partial E}{\partial w_i} = 0$ より、求める連立一次方程式は：
$$
\sum_{j=0}^M \tilde{A}_{ij} w_j = T_i \tag{Ans}
$$
ここで、
$$
\tilde{A}_{ij} = A_{ij} + \lambda \delta_{ij} = \sum_{n=1}^N (x_n)^{i+j} + \lambda \delta_{ij}
$$
行列形式では $(\mathbf{A} + \lambda \mathbf{I}) \mathbf{w} = \mathbf{T}$ となります。 $\blacksquare$


In [3]:
# Exercise 4.2 の数値検証
lam = 0.5
w_reg, A_reg = solve_regularized_polynomial_equations(x_sample, t_sample, M, lam=lam)

# 勾配が厳密に 0 になることを確認
Phi_ex2 = np.vander(x_sample, M + 1, increasing=True)
grad_reg = (Phi_ex2.T @ Phi_ex2 + lam * np.eye(M + 1)) @ w_reg - Phi_ex2.T @ t_sample

print("正則化解 w_reg:", w_reg)
print("正則化解での勾配ノルム:", la.norm(grad_reg))
assert la.norm(grad_reg) < 1e-12
print("Exercise 4.2: 検証成功！")


正則化解 w_reg: [ 0.83656121 -1.92542943  1.32978137]
正則化解での勾配ノルム: 1.0658141036401503e-14
Exercise 4.2: 検証成功！


---
## Exercise 4.3 (?): $\tanh$ とロジスティック・シグモイドの等価性とパラメータ変換

### 問題文
Show that the $\tanh$ function defined by
$$
\tanh(a) = \frac{e^a - e^{-a}}{e^a + e^{-a}} \tag{4.55}
$$
and the logistic sigmoid function defined by (4.6) are related by
$$
\tanh(a) = 2\sigma(2a) - 1 \tag{4.56}
$$
Hence, show that a general linear combination of logistic sigmoid functions of the form
$$
y(x, \mathbf{w}) = w_0 + \sum_{j=1}^M w_j \sigma\left(\frac{x - \mu_j}{s}\right) \tag{4.57}
$$
is equivalent to a linear combination of $\tanh$ functions of the form
$$
y(x, \mathbf{u}) = u_0 + \sum_{j=1}^M u_j \tanh\left(\frac{x - \mu_j}{2s}\right) \tag{4.58}
$$
and find expressions to relate the new parameters $\{u_0, u_1, \dots, u_M\}$ to the original parameters $\{w_0, w_1, \dots, w_M\}$.

---

### 数学的証明
#### 1. 関係式 (4.56) の証明
ロジスティック・シグモイド関数は $\sigma(z) = \frac{1}{1 + e^{-z}} = \frac{e^z}{e^z + 1}$ です。
$z = 2a$ と置くと：
$$
2\sigma(2a) - 1 = \frac{2}{1 + e^{-2a}} - 1 = \frac{2 - (1 + e^{-2a})}{1 + e^{-2a}} = \frac{1 - e^{-2a}}{1 + e^{-2a}}
$$
分子・分母に $e^a$ を掛けると：
$$
\frac{e^a (1 - e^{-2a})}{e^a (1 + e^{-2a})} = \frac{e^a - e^{-a}}{e^a + e^{-a}} = \tanh(a)
$$
よって $\tanh(a) = 2\sigma(2a) - 1$ が示されました。

#### 2. 線形結合の等価性とパラメータ変換
式 (4.56) を変形すると、シグモイド関数は $\tanh$ を用いて次のように表せます：
$$
\sigma(2a) = \frac{1}{2}\tanh(a) + \frac{1}{2} \implies \sigma(z) = \frac{1}{2}\tanh\left(\frac{z}{2}\right) + \frac{1}{2}
$$
ここで $z = \frac{x - \mu_j}{s}$ を代入すると：
$$
\sigma\left(\frac{x - \mu_j}{s}\right) = \frac{1}{2}\tanh\left(\frac{x - \mu_j}{2s}\right) + \frac{1}{2}
$$
これを式 (4.57) に代入します：
$$
\begin{aligned}
y(x, \mathbf{w}) &= w_0 + \sum_{j=1}^M w_j \left\{ \frac{1}{2}\tanh\left(\frac{x - \mu_j}{2s}\right) + \frac{1}{2} \right\} \\
&= \left( w_0 + \frac{1}{2}\sum_{j=1}^M w_j \right) + \sum_{j=1}^M \left(\frac{1}{2} w_j\right) \tanh\left(\frac{x - \mu_j}{2s}\right)
\end{aligned}
$$
これを式 (4.58) の $y(x, \mathbf{u}) = u_0 + \sum_{j=1}^M u_j \tanh\left(\frac{x - \mu_j}{2s}\right)$ と比較すると、係数は一対一に対応し：
$$
\begin{aligned}
u_j &= \frac{1}{2} w_j \quad (j = 1, \dots, M) \\
u_0 &= w_0 + \frac{1}{2}\sum_{j=1}^M w_j
\end{aligned} \tag{Ans}
$$
となります。したがって両者は厳密に等価です。 $\blacksquare$


In [4]:
# Exercise 4.3 の数値検証
w0_orig = 2.4
w_orig = np.array([1.2, -0.8, 3.1, -1.5])
mu_orig = np.array([-1.0, -0.3, 0.4, 1.2])
s_orig = 0.4

u0_conv, u_conv, mu_conv, s_conv = sigmoid_to_tanh_params(w0_orig, w_orig, mu_orig, s_orig)

# 任意の x 点上で出力を比較
x_test_pts = np.linspace(-2, 2, 100)
y_sig_out = eval_sigmoid_network(x_test_pts, w0_orig, w_orig, mu_orig, s_orig)
y_tanh_out = eval_tanh_network(x_test_pts, u0_conv, u_conv, mu_conv, s_conv)

max_diff = np.max(np.abs(y_sig_out - y_tanh_out))
print(f"シグモイド出力と tanh 出力の最大絶対誤差: {max_diff:.2e}")
assert max_diff < 1e-14
print("Exercise 4.3: 検証成功！")


シグモイド出力と tanh 出力の最大絶対誤差: 1.78e-15
Exercise 4.3: 検証成功！


---
## Exercise 4.4 (? ? ?): 射影行列と最小二乗法の直交幾何学

### 問題文
Show that the matrix
$$
\mathbf{P} = \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T \tag{4.59}
$$
takes any vector $\mathbf{v}$ and projects it onto the space spanned by the columns of $\boldsymbol{\Phi}$. Use this result to show that the least-squares solution (4.14) corresponds to an orthogonal projection of the vector $\mathbf{t}$ onto the manifold $\mathcal{S}$, as shown in Figure 4.3.

---

### 数学的証明
計画行列 $\boldsymbol{\Phi}$ は $N \times M$ 行列であり、その列ベクトルを $\boldsymbol{\phi}_1, \dots, \boldsymbol{\phi}_M \in \mathbb{R}^N$ とします。
列空間（部分空間）を $\mathcal{S} = \text{span}(\boldsymbol{\phi}_1, \dots, \boldsymbol{\phi}_M)$ と定義します。

#### 1. 射影行列の定義（冪等性と対称性）
ある線形変換 $\mathbf{P}$ が直交射影行列であるための必要十分条件は、**対称性**（$\mathbf{P}^T = \mathbf{P}$）と**冪等性**（$\mathbf{P}^2 = \mathbf{P}$）を満たすことです。
- **対称性**:
  $$
  \mathbf{P}^T = \left( \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T \right)^T = (\boldsymbol{\Phi}^T)^T \left((\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}\right)^T \boldsymbol{\Phi}^T = \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T = \mathbf{P}
  $$
- **冪等性**:
  $$
  \mathbf{P}^2 = \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \underbrace{\boldsymbol{\Phi}^T \boldsymbol{\Phi} (\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}}_{= \mathbf{I}} \boldsymbol{\Phi}^T = \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T = \mathbf{P}
  $$

#### 2. 任意のベクトル $\mathbf{v}$ の像が $\mathcal{S}$ に属すること
任意のベクトル $\mathbf{v} \in \mathbb{R}^N$ に対して：
$$
\mathbf{P}\mathbf{v} = \boldsymbol{\Phi} \underbrace{\left((\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T \mathbf{v}\right)}_{\mathbf{a} \in \mathbb{R}^M} = \boldsymbol{\Phi}\mathbf{a} = \sum_{j=1}^M a_j \boldsymbol{\phi}_j \in \mathcal{S}
$$
よって $\mathbf{P}\mathbf{v}$ は常に $\boldsymbol{\Phi}$ の列の線形結合であり、部分空間 $\mathcal{S}$ に属します。

#### 3. 残差ベクトルの直交性
残差ベクトルを $\mathbf{r} = (\mathbf{I} - \mathbf{P})\mathbf{v}$ と置きます。
部分空間 $\mathcal{S}$ の任意の基底ベクトル $\boldsymbol{\phi}_j$ との直交性を確認するため、$\boldsymbol{\Phi}^T \mathbf{r}$ を計算します：
$$
\boldsymbol{\Phi}^T \mathbf{r} = \boldsymbol{\Phi}^T (\mathbf{I} - \mathbf{P})\mathbf{v} = \boldsymbol{\Phi}^T \mathbf{v} - \underbrace{\boldsymbol{\Phi}^T \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}}_{= \mathbf{I}} \boldsymbol{\Phi}^T \mathbf{v} = \boldsymbol{\Phi}^T \mathbf{v} - \boldsymbol{\Phi}^T \mathbf{v} = \mathbf{0}
$$
したがって、残差ベクトル $\mathbf{r} = \mathbf{v} - \mathbf{P}\mathbf{v}$ は、部分空間 $\mathcal{S}$ のすべての基底ベクトルと直交（直交補空間 $\mathcal{S}^\perp$ に属する）します。

#### 4. 最小二乗解との一致
式 (4.14) の最小二乗解 $\mathbf{w}_{\mathrm{ML}} = (\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T \mathbf{t}$ を用いたモデルの予測ベクトル $\mathbf{y}$ は：
$$
\mathbf{y} = \boldsymbol{\Phi}\mathbf{w}_{\mathrm{ML}} = \boldsymbol{\Phi}(\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T \mathbf{t} = \mathbf{P}\mathbf{t}
$$
となります。
これは、観測ターゲットベクトル $\mathbf{t} \in \mathbb{R}^N$ を部分空間 $\mathcal{S}$ に直交射影したベクトルそのものであり、幾何学的に $\|\mathbf{t} - \mathbf{y}\|$ を最小化する点が直交射影によって与えられることが示されました（Figure 4.3）。 $\blacksquare$


In [5]:
# Exercise 4.4 の数値検証
rng = np.random.RandomState(42)
N, M = 20, 3
Phi_mat = rng.randn(N, M)
v_vec = rng.randn(N)

P_mat = compute_projection_matrix(Phi_mat)

# 1. 対称性と冪等性の確認
assert np.allclose(P_mat.T, P_mat, atol=1e-12), "P must be symmetric"
assert np.allclose(P_mat @ P_mat, P_mat, atol=1e-12), "P must be idempotent"

# 2. 射影の直交性 Phi^T (I - P) v == 0
residual_vec = (np.eye(N) - P_mat) @ v_vec
orthogonal_check = Phi_mat.T @ residual_vec
print("直交性残差ノルム ||Phi^T (I - P) v||:", la.norm(orthogonal_check))
assert la.norm(orthogonal_check) < 1e-12

print("Exercise 4.4: 検証成功！")


直交性残差ノルム ||Phi^T (I - P) v||: 1.4432899320127035e-15
Exercise 4.4: 検証成功！


---
## Exercise 4.5 (?): 重み付き最小二乗法 (Weighted Least Squares)

### 問題文
Consider a data set in which each data point $t_n$ is associated with a weighting factor $r_n > 0$, so that the sum-of-squares error function becomes
$$
E_D(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N r_n \{t_n - \mathbf{w}^T \boldsymbol{\phi}(x_n)\}^2 \tag{4.60}
$$
Find an expression for the solution $\mathbf{w}^*$ that minimizes this error function. Give two alternative interpretations of the weighted sum-of-squares error function in terms of (i) data-dependent noise variance and (ii) replicated data points.

---

### 数学的証明
#### 1. 最適解 $\mathbf{w}^*$ の導出
誤差関数を行列形式で表します。重み対角行列を $\mathbf{R} = \text{diag}(r_1, \dots, r_N)$ と定義すると：
$$
E_D(\mathbf{w}) = \frac{1}{2} (\mathbf{t} - \boldsymbol{\Phi}\mathbf{w})^T \mathbf{R} (\mathbf{t} - \boldsymbol{\Phi}\mathbf{w})
$$
これを $\mathbf{w}$ に関して微分します：
$$
\nabla_{\mathbf{w}} E_D = - \boldsymbol{\Phi}^T \mathbf{R} (\mathbf{t} - \boldsymbol{\Phi}\mathbf{w}) = \boldsymbol{\Phi}^T \mathbf{R} \boldsymbol{\Phi}\mathbf{w} - \boldsymbol{\Phi}^T \mathbf{R}\mathbf{t}
$$
停留条件 $\nabla_{\mathbf{w}} E_D = \mathbf{0}$ より：
$$
\boldsymbol{\Phi}^T \mathbf{R} \boldsymbol{\Phi}\mathbf{w}^* = \boldsymbol{\Phi}^T \mathbf{R}\mathbf{t} \implies \mathbf{w}^* = (\boldsymbol{\Phi}^T \mathbf{R} \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T \mathbf{R}\mathbf{t} \tag{Ans}
$$

#### 2. 解釈 (i): データ依存ノイズ分散（不均一分散 / Heteroscedasticity）
各データ点 $n$ のノイズ分散が異なり、$t_n \sim \mathcal{N}(\mathbf{w}^T \boldsymbol{\phi}(x_n), \sigma_n^2)$ に従うと仮定します。
独立標本に対する対数尤度関数は：
$$
\ln p(\mathbf{t}|\mathbf{X}, \mathbf{w}) = -\frac{N}{2}\ln(2\pi) - \sum_{n=1}^N \ln \sigma_n - \frac{1}{2}\sum_{n=1}^N \frac{1}{\sigma_n^2} \{t_n - \mathbf{w}^T \boldsymbol{\phi}(x_n)\}^2
$$
これを $\mathbf{w}$ に関して最大化することは：
$$
\frac{1}{2}\sum_{n=1}^N \frac{1}{\sigma_n^2} \{t_n - \mathbf{w}^T \boldsymbol{\phi}(x_n)\}^2
$$
を最小化することと等価です。したがって、$r_n = \frac{1}{\sigma_n^2}$（精度パラメータ）とおけば、重み付き最小二乗法は**ノイズ分散が観測点ごとに異なる正規分布の最尤推定**と完全に一致します。

#### 3. 解釈 (ii): データ点の複製（Replicated Data Points）
各 $r_n$ が正の整数である場合を考えます。
もしデータ点 $(\mathbf{x}_n, t_n)$ がデータセット中に独立に $r_n$ 回重複して観測されたとすると、通常の二乗和誤差は：
$$
\frac{1}{2} \sum_{n=1}^N \sum_{k=1}^{r_n} \{t_n - \mathbf{w}^T \boldsymbol{\phi}(x_n)\}^2 = \frac{1}{2} \sum_{n=1}^N r_n \{t_n - \mathbf{w}^T \boldsymbol{\phi}(x_n)\}^2
$$
となり、式 (4.60) と完全に一致します。したがって、重み $r_n$ は**データ点の頻度（観測回数）**としての自然な解釈を持ちます。 $\blacksquare$


In [6]:
# Exercise 4.5 の数値検証
Phi_wls = np.array([[1.0, 0.2], [1.0, 0.8], [1.0, 1.5]])
t_wls = np.array([1.0, 2.5, 3.2])
weights_int = np.array([3.0, 1.0, 2.0]) # 整数重み

# WLS 解
w_wls_sol = solve_weighted_least_squares(Phi_wls, t_wls, weights_int)

# 複製データセットを作成
Phi_replicated = np.vstack([Phi_wls[0]] * 3 + [Phi_wls[1]] * 1 + [Phi_wls[2]] * 2)
t_replicated = np.hstack([t_wls[0]] * 3 + [t_wls[1]] * 1 + [t_wls[2]] * 2)
w_rep_sol = la.solve(Phi_replicated.T @ Phi_replicated, Phi_replicated.T @ t_replicated, assume_a='pos')

print("WLS 解:      ", w_wls_sol)
print("複製データ解:", w_rep_sol)
assert np.allclose(w_wls_sol, w_rep_sol, atol=1e-12)
print("Exercise 4.5: 検証成功！")


WLS 解:       [0.73065574 1.70819672]
複製データ解: [0.73065574 1.70819672]
Exercise 4.5: 検証成功！


---
## Exercise 4.6 (?): 正則化二乗和誤差の厳密な停留解の導出

### 問題文
By setting the gradient of (4.26) with respect to $\mathbf{w}$ to zero, show that the exact minimum of the regularized sum-of-squares error function for linear regression is given by (4.27).

---

### 数学的証明
正則化二乗和誤差関数 (4.26) は：
$$
E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \{t_n - \mathbf{w}^T \boldsymbol{\phi}(x_n)\}^2 + \frac{\lambda}{2} \mathbf{w}^T \mathbf{w} \tag{4.26}
$$
計画行列 $\boldsymbol{\Phi}$ とターゲットベクトル $\mathbf{t}$ を用いて展開すると：
$$
E(\mathbf{w}) = \frac{1}{2} (\mathbf{t} - \boldsymbol{\Phi}\mathbf{w})^T (\mathbf{t} - \boldsymbol{\Phi}\mathbf{w}) + \frac{\lambda}{2} \mathbf{w}^T \mathbf{w} = \frac{1}{2} \mathbf{t}^T \mathbf{t} - \mathbf{w}^T \boldsymbol{\Phi}^T \mathbf{t} + \frac{1}{2} \mathbf{w}^T (\boldsymbol{\Phi}^T \boldsymbol{\Phi} + \lambda \mathbf{I}) \mathbf{w}
$$
ベクトル $\mathbf{w}$ に関して勾配をとります：
$$
\nabla_{\mathbf{w}} E(\mathbf{w}) = - \boldsymbol{\Phi}^T \mathbf{t} + (\boldsymbol{\Phi}^T \boldsymbol{\Phi} + \lambda \mathbf{I}) \mathbf{w}
$$
これをゼロとおきます（停留条件）：
$$
(\boldsymbol{\Phi}^T \boldsymbol{\Phi} + \lambda \mathbf{I}) \mathbf{w} = \boldsymbol{\Phi}^T \mathbf{t}
$$
$\lambda > 0$ であるとき、$\boldsymbol{\Phi}^T \boldsymbol{\Phi}$ は半正定値、$\lambda \mathbf{I}$ は正定値であるため、行列 $(\lambda \mathbf{I} + \boldsymbol{\Phi}^T \boldsymbol{\Phi})$ は常に厳密に正定値（可逆）です。
したがって両辺に逆行列を掛けることにより、式 (4.27) が得られます：
$$
\mathbf{w} = (\lambda \mathbf{I} + \boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T \mathbf{t} \tag{4.27}
$$
またヘッセ行列は $\nabla_{\mathbf{w}}^2 E(\mathbf{w}) = \boldsymbol{\Phi}^T \boldsymbol{\Phi} + \lambda \mathbf{I} \succ \mathbf{0}$ であり、この解は厳密な大域的最小値を与えます。 $\blacksquare$


In [7]:
# Exercise 4.6 の数値検証
Phi_ex6 = rng.randn(15, 4)
t_ex6 = rng.randn(15)
lam_ex6 = 1.25

w_ex6 = la.solve(lam_ex6 * np.eye(4) + Phi_ex6.T @ Phi_ex6, Phi_ex6.T @ t_ex6, assume_a='pos')
grad_ex6 = (Phi_ex6.T @ Phi_ex6 + lam_ex6 * np.eye(4)) @ w_ex6 - Phi_ex6.T @ t_ex6

print("勾配ノルム:", la.norm(grad_ex6))
assert la.norm(grad_ex6) < 1e-12
print("Exercise 4.6: 検証成功！")


勾配ノルム: 9.222205069512407e-16
Exercise 4.6: 検証成功！


---
## Exercise 4.7 (? ?): 多変量線形回帰と一般共分散行列の最尤推定

### 問題文
Consider a linear basis function regression model for a multivariate target variable $\mathbf{t}$ having a Gaussian distribution of the form
$$
p(\mathbf{t}|\mathbf{W}, \boldsymbol{\Sigma}) = \mathcal{N}(\mathbf{t}|\mathbf{y}(\mathbf{x}, \mathbf{W}), \boldsymbol{\Sigma}) \tag{4.61}
$$
where
$$
\mathbf{y}(\mathbf{x}, \mathbf{W}) = \mathbf{W}^T \boldsymbol{\phi}(\mathbf{x}) \tag{4.62}
$$
together with a training data set comprising input basis vectors $\boldsymbol{\phi}(\mathbf{x}_n)$ and corresponding target vectors $\mathbf{t}_n$, with $n = 1, \dots, N$. Show that the maximum likelihood solution $\mathbf{W}_{\mathrm{ML}}$ for the parameter matrix $\mathbf{W}$ has the property that each column is given by an expression of the form (4.14), which was the solution for an isotropic noise distribution. Note that this is independent of the covariance matrix $\boldsymbol{\Sigma}$. Show that the maximum likelihood solution for $\boldsymbol{\Sigma}$ is given by
$$
\boldsymbol{\Sigma} = \frac{1}{N} \sum_{n=1}^N \{\mathbf{t}_n - \mathbf{W}_{\mathrm{ML}}^T \boldsymbol{\phi}(\mathbf{x}_n)\} \{\mathbf{t}_n - \mathbf{W}_{\mathrm{ML}}^T \boldsymbol{\phi}(\mathbf{x}_n)\}^T \tag{4.63}
$$

---

### 数学的証明
各観測は独立であるため、データ全体の対数尤度関数は：
$$
\ln p(\mathbf{T}|\mathbf{X}, \mathbf{W}, \boldsymbol{\Sigma}) = -\frac{NK}{2}\ln(2\pi) - \frac{N}{2}\ln|\boldsymbol{\Sigma}| - \frac{1}{2}\sum_{n=1}^N (\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)^T \boldsymbol{\Sigma}^{-1} (\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)
$$
ここでトレースの巡回対称性 $\mathbf{a}^T \mathbf{B} \mathbf{a} = \text{Tr}(\mathbf{a}^T \mathbf{B} \mathbf{a}) = \text{Tr}(\mathbf{B} \mathbf{a} \mathbf{a}^T)$ を用いると、二次形式の和は：
$$
\sum_{n=1}^N (\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)^T \boldsymbol{\Sigma}^{-1} (\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n) = \text{Tr}\left( \boldsymbol{\Sigma}^{-1} \sum_{n=1}^N (\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)(\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)^T \right)
$$
行列形式で $\mathbf{T} \in \mathbb{R}^{N \times K}$, $\boldsymbol{\Phi} \in \mathbb{R}^{N \times M}$ と表記すると：
$$
\sum_{n=1}^N (\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)(\mathbf{t}_n - \mathbf{W}^T \boldsymbol{\phi}_n)^T = (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W})^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W})
$$
したがって、対数尤度関数は次のように表せます：
$$
\ln p = -\frac{NK}{2}\ln(2\pi) - \frac{N}{2}\ln|\boldsymbol{\Sigma}| - \frac{1}{2}\text{Tr}\Big( \boldsymbol{\Sigma}^{-1} (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W})^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}) \Big)
$$

#### 1. $\mathbf{W}$ に関する最大化
$\mathbf{W}$ に関する偏微分を計算します：
$$
\frac{\partial}{\partial \mathbf{W}} \text{Tr}\Big( \boldsymbol{\Sigma}^{-1} (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W})^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}) \Big) = -2 \boldsymbol{\Phi}^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}) \boldsymbol{\Sigma}^{-1}
$$
これを $\mathbf{0}$ とおくと：
$$
\boldsymbol{\Phi}^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}) \boldsymbol{\Sigma}^{-1} = \mathbf{O}
$$
共分散行列 $\boldsymbol{\Sigma}$ は正定値・可逆であるため、両辺の右から $\boldsymbol{\Sigma}$ を掛けることで $\boldsymbol{\Sigma}^{-1}$ が完全に消去されます：
$$
\boldsymbol{\Phi}^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}) = \mathbf{O} \implies \boldsymbol{\Phi}^T \boldsymbol{\Phi}\mathbf{W} = \boldsymbol{\Phi}^T \mathbf{T}
$$
したがって、最尤解 $\mathbf{W}_{\mathrm{ML}}$ は：
$$
\mathbf{W}_{\mathrm{ML}} = (\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^T \mathbf{T}
$$
となり、**共分散行列 $\boldsymbol{\Sigma}$ の値に全く依存せず、等方性ガウスノイズの場合の解と完全に一致**します！
列ごとに見れば、第 $k$ 列は $\mathbf{w}_k = (\boldsymbol{\Phi}^T \boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T \mathbf{t}_{:, k}$ であり、式 (4.14) そのものです。

#### 2. $\boldsymbol{\Sigma}$ に関する最大化
精度行列 $\boldsymbol{\Lambda} = \boldsymbol{\Sigma}^{-1}$ に関して対数尤度を微分します（第2章・第3章の正規分布の最尤推定と同様）：
$$
\ln p = \frac{N}{2}\ln|\boldsymbol{\Lambda}| - \frac{1}{2}\text{Tr}\Big( \boldsymbol{\Lambda} \mathbf{S} \Big) + \text{const}, \quad \mathbf{S} = (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}_{\mathrm{ML}})^T (\mathbf{T} - \boldsymbol{\Phi}\mathbf{W}_{\mathrm{ML}})
$$
$\boldsymbol{\Lambda}$ に関する微分をゼロとおくと：
$$
\frac{N}{2} \boldsymbol{\Lambda}^{-1} - \frac{1}{2} \mathbf{S} = \mathbf{O} \implies \boldsymbol{\Sigma}_{\mathrm{ML}} = \frac{1}{N}\mathbf{S} = \frac{1}{N} \sum_{n=1}^N \{\mathbf{t}_n - \mathbf{W}_{\mathrm{ML}}^T \boldsymbol{\phi}_n\} \{\mathbf{t}_n - \mathbf{W}_{\mathrm{ML}}^T \boldsymbol{\phi}_n\}^T
$$
となり、式 (4.63) が得られました。 $\blacksquare$


In [8]:
# Exercise 4.7 の数値検証
Phi_multi = rng.randn(30, 4)
T_multi = rng.randn(30, 3)

W_ml_sol, Sigma_ml_sol = estimate_multivariate_regression_mle(Phi_multi, T_multi)

# 各列が独立な通常の最小二乗解と一致するか検証
for k in range(3):
    w_k_single = la.solve(Phi_multi.T @ Phi_multi, Phi_multi.T @ T_multi[:, k], assume_a='pos')
    assert np.allclose(W_ml_sol[:, k], w_k_single, atol=1e-12)

print("W_ML shape:", W_ml_sol.shape)
print("Sigma_ML shape:", Sigma_ml_sol.shape)
print("Sigma_ML 固有値:", la.eigvalsh(Sigma_ml_sol))
assert np.all(la.eigvalsh(Sigma_ml_sol) > 0)
print("Exercise 4.7: 検証成功！")


W_ML shape: (4, 3)
Sigma_ML shape: (3, 3)
Sigma_ML 固有値: [0.3777833  0.52355886 0.68998736]
Exercise 4.7: 検証成功！


---
## Exercises 4.8, 4.9, 4.10: 多変量二乗損失と条件付き期待値の最適性

### 問題文 (Ex 4.8)
Consider the generalization of the squared-loss function (4.35) for a single target variable $t$ to multiple target variables described by the vector $\mathbf{t}$ given by
$$
\mathbb{E}[L(\mathbf{t}, \mathbf{f}(\mathbf{x}))] = \iint \|\mathbf{f}(\mathbf{x}) - \mathbf{t}\|^2 p(\mathbf{x}, \mathbf{t}) \, d\mathbf{x} \, d\mathbf{t} \tag{4.64}
$$
Using the calculus of variations, show that the function $\mathbf{f}(\mathbf{x})$ for which this expected loss is minimized is given by
$$
\mathbf{f}(\mathbf{x}) = \mathbb{E}_{\mathbf{t}}[\mathbf{t}|\mathbf{x}] \tag{4.65}
$$

### 問題文 (Ex 4.9 & 4.10)
- **Ex 4.9**: By expansion of the square in (4.64), derive a result analogous to (4.39) and, hence, show that the function $\mathbf{f}(\mathbf{x})$ that minimizes the expected squared loss for a vector $\mathbf{t}$ of target variables is again given by the conditional expectation of $\mathbf{t}$ in the form (4.65).
- **Ex 4.10**: Rederive the result (4.65) by first expanding (4.64) analogous to (4.39).

---

### 数学的証明
#### 1. 変分法による解法 (Exercise 4.8)
$\mathbf{f}(\mathbf{x})$ の任意の微小変分 $\delta \mathbf{f}(\mathbf{x})$ を考えます：
$$
\begin{aligned}
\delta \mathbb{E}[L] &= \iint 2 (\mathbf{f}(\mathbf{x}) - \mathbf{t})^T \delta \mathbf{f}(\mathbf{x}) \, p(\mathbf{x}, \mathbf{t}) \, d\mathbf{x} \, d\mathbf{t} \\
&= 2 \int \delta \mathbf{f}(\mathbf{x})^T \left\{ \int (\mathbf{f}(\mathbf{x}) - \mathbf{t}) p(\mathbf{x}, \mathbf{t}) \, d\mathbf{t} \right\} d\mathbf{x}
\end{aligned}
$$
任意の変分 $\delta \mathbf{f}(\mathbf{x})$ に対して $\delta \mathbb{E}[L] = 0$ となるためには、括弧内の積分がすべての $\mathbf{x}$ についてゼロでなければなりません：
$$
\int (\mathbf{f}(\mathbf{x}) - \mathbf{t}) p(\mathbf{x}, \mathbf{t}) \, d\mathbf{t} = \mathbf{0}
$$
乗法定理 $p(\mathbf{x}, \mathbf{t}) = p(\mathbf{t}|\mathbf{x}) p(\mathbf{x})$ を代入すると：
$$
p(\mathbf{x}) \left\{ \mathbf{f}(\mathbf{x}) \int p(\mathbf{t}|\mathbf{x}) \, d\mathbf{t} - \int \mathbf{t} p(\mathbf{t}|\mathbf{x}) \, d\mathbf{t} \right\} = \mathbf{0}
$$
$\int p(\mathbf{t}|\mathbf{x}) d\mathbf{t} = 1$ より、直ちに次式が得られます：
$$
\mathbf{f}(\mathbf{x}) = \int \mathbf{t} p(\mathbf{t}|\mathbf{x}) \, d\mathbf{t} = \mathbb{E}_{\mathbf{t}}[\mathbf{t}|\mathbf{x}] \tag{4.65}
$$

#### 2. 二乗展開による直交分解 (Exercises 4.9 & 4.10)
$\mathbf{h}(\mathbf{x}) = \mathbb{E}[\mathbf{t}|\mathbf{x}]$ と定義します。ノルム二乗の中に $\mathbf{h}(\mathbf{x})$ を足して引きます：
$$
\|\mathbf{f}(\mathbf{x}) - \mathbf{t}\|^2 = \|(\mathbf{f}(\mathbf{x}) - \mathbf{h}(\mathbf{x})) + (\mathbf{h}(\mathbf{x}) - \mathbf{t})\|^2 = \|\mathbf{f}(\mathbf{x}) - \mathbf{h}(\mathbf{x})\|^2 + \|\mathbf{h}(\mathbf{x}) - \mathbf{t}\|^2 + 2 (\mathbf{f}(\mathbf{x}) - \mathbf{h}(\mathbf{x}))^T (\mathbf{h}(\mathbf{x}) - \mathbf{t})
$$
これを期待損失 (4.64) に代入し、交差項の $\mathbf{t}$ に関する積分を計算します：
$$
\int (\mathbf{f}(\mathbf{x}) - \mathbf{h}(\mathbf{x}))^T (\mathbf{h}(\mathbf{x}) - \mathbf{t}) p(\mathbf{t}|\mathbf{x}) \, d\mathbf{t} = (\mathbf{f}(\mathbf{x}) - \mathbf{h}(\mathbf{x}))^T \underbrace{\int (\mathbf{h}(\mathbf{x}) - \mathbf{t}) p(\mathbf{t}|\mathbf{x}) \, d\mathbf{t}}_{= \mathbf{h}(\mathbf{x}) - \mathbf{h}(\mathbf{x}) = \mathbf{0}} = 0
$$
交差項は厳密に消失します！したがって式 (4.39) の多変量版が得られます：
$$
\mathbb{E}[L] = \int \|\mathbf{f}(\mathbf{x}) - \mathbb{E}[\mathbf{t}|\mathbf{x}]\|^2 p(\mathbf{x}) \, d\mathbf{x} + \iint \|\mathbb{E}[\mathbf{t}|\mathbf{x}] - \mathbf{t}\|^2 p(\mathbf{x}, \mathbf{t}) \, d\mathbf{x} \, d\mathbf{t}
$$
第2項は $\mathbf{f}(\mathbf{x})$ に依存しない定数（固有ノイズ分散の総和）であり、第1項は $\|\cdot\|^2 \ge 0$ より非負です。
したがって、期待損失を最小化する唯一の解は第1項をゼロにする $\mathbf{f}(\mathbf{x}) = \mathbb{E}[\mathbf{t}|\mathbf{x}]$ であり、(4.65) が再導出されました。 $\blacksquare$


In [9]:
# Exercises 4.8 - 4.10 の数値検証
# 条件付き二変量ターゲット t | x
x_val = 0.7
h_true = np.array([np.sin(x_val), np.cos(x_val)])
Sigma_true = np.array([[0.3, 0.1], [0.1, 0.2]])

samples_t = rng.multivariate_normal(h_true, Sigma_true, size=50000)

loss_optimal = np.mean(np.sum((samples_t - h_true[None, :]) ** 2, axis=1))

# 意図的にずらした予測値
perturbed_pred = h_true + np.array([0.5, -0.4])
loss_perturbed = np.mean(np.sum((samples_t - perturbed_pred[None, :]) ** 2, axis=1))

excess_loss = loss_perturbed - loss_optimal
theoretical_excess = np.sum((perturbed_pred - h_true) ** 2)

print(f"最適予測 h(x) の損失:    {loss_optimal:.4f}")
print(f"ずらした予測の損失:       {loss_perturbed:.4f}")
print(f"超過損失 (シミュレーション): {excess_loss:.4f}")
print(f"理論的超過損失 ||f - h||^2: {theoretical_excess:.4f}")

assert loss_perturbed > loss_optimal
assert np.isclose(excess_loss, theoretical_excess, rtol=0.03)
print("Exercises 4.8 - 4.10: 検証成功！")


最適予測 h(x) の損失:    0.4987
ずらした予測の損失:       0.9099
超過損失 (シミュレーション): 0.4112
理論的超過損失 ||f - h||^2: 0.4100
Exercises 4.8 - 4.10: 検証成功！


---
## Exercise 4.11 (? ?): 一般化ガウス分布と $L_q$ 損失関数

### 問題文
The following distribution
$$
p(x|\sigma^2, q) = \frac{q}{2(2\sigma^2)^{1/q}\Gamma(1/q)} \exp\left( -\frac{|x|^q}{2\sigma^2} \right) \tag{4.66}
$$
is a generalization of the univariate Gaussian distribution. Here $\Gamma(x)$ is the gamma function defined by (4.67).
Show that this distribution is normalized so that $\int_{-\infty}^\infty p(x|\sigma^2, q) dx = 1$, and that it reduces to the Gaussian when $q = 2$.
Consider a regression model in which the target variable is given by $t = y(\mathbf{x}, \mathbf{w}) + \epsilon$ and $\epsilon$ is a random noise variable drawn from (4.66). Show that the log likelihood function over $\mathbf{w}$ and $\sigma^2$ is given by (4.69):
$$
\ln p(\mathbf{t}|\mathbf{X}, \mathbf{w}, \sigma^2) = -\frac{1}{2\sigma^2} \sum_{n=1}^N |y(\mathbf{x}_n, \mathbf{w}) - t_n|^q - \frac{N}{q}\ln(2\sigma^2) + \text{const} \tag{4.69}
$$

---

### 数学的証明
#### 1. 規格化条件の証明
被積分関数は $x$ に関して対称（偶関数）であるため：
$$
I = \int_{-\infty}^\infty p(x|\sigma^2, q) \, dx = 2 \frac{q}{2(2\sigma^2)^{1/q}\Gamma(1/q)} \int_0^\infty \exp\left( -\frac{x^q}{2\sigma^2} \right) dx
$$
変数変換を行います。$u = \frac{x^q}{2\sigma^2}$ と置くと：
$$
x = (2\sigma^2 u)^{1/q} = (2\sigma^2)^{1/q} u^{1/q} \implies dx = (2\sigma^2)^{1/q} \frac{1}{q} u^{\frac{1}{q} - 1} du
$$
これを積分に代入します：
$$
I = \frac{q}{(2\sigma^2)^{1/q}\Gamma(1/q)} \int_0^\infty e^{-u} (2\sigma^2)^{1/q} \frac{1}{q} u^{\frac{1}{q} - 1} du = \frac{1}{\Gamma(1/q)} \int_0^\infty u^{\frac{1}{q} - 1} e^{-u} du
$$
ガンマ関数の定義 $\Gamma(z) = \int_0^\infty u^{z-1} e^{-u} du$ より、右辺の積分は厳密に $\Gamma(1/q)$ です。
したがって：
$$
I = \frac{\Gamma(1/q)}{\Gamma(1/q)} = 1
$$
となり、任意の $q > 0, \sigma^2 > 0$ に対して規格化されていることが示されました。

#### 2. $q = 2$ のときガウス分布に帰着すること
$q = 2$ を代入すると、$\Gamma(1/2) = \sqrt{\pi}$、$(2\sigma^2)^{1/2} = \sqrt{2\sigma^2}$ であるため：
$$
p(x|\sigma^2, 2) = \frac{2}{2 \sqrt{2\sigma^2} \sqrt{\pi}} \exp\left( -\frac{x^2}{2\sigma^2} \right) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{x^2}{2\sigma^2} \right) = \mathcal{N}(x|0, \sigma^2)
$$
となり、標準的な 1 次元正規分布に完全に一致します。

#### 3. 対数尤度関数 (4.69) の導出
ノイズ $\epsilon_n = t_n - y(\mathbf{x}_n, \mathbf{w})$ が独立に分布 (4.66) に従うため：
$$
p(t_n|\mathbf{x}_n, \mathbf{w}, \sigma^2) = \frac{q}{2(2\sigma^2)^{1/q}\Gamma(1/q)} \exp\left( -\frac{|y(\mathbf{x}_n, \mathbf{w}) - t_n|^q}{2\sigma^2} \right)
$$
全データセットに対する対数尤度関数は：
$$
\begin{aligned}
\ln p(\mathbf{t}|\mathbf{X}, \mathbf{w}, \sigma^2) &= \sum_{n=1}^N \ln p(t_n|\mathbf{x}_n, \mathbf{w}, \sigma^2) \\
&= \sum_{n=1}^N \left\{ \ln q - \ln 2 - \frac{1}{q}\ln(2\sigma^2) - \ln\Gamma(1/q) - \frac{|y(\mathbf{x}_n, \mathbf{w}) - t_n|^q}{2\sigma^2} \right\} \\
&= -\frac{1}{2\sigma^2} \sum_{n=1}^N |y(\mathbf{x}_n, \mathbf{w}) - t_n|^q - \frac{N}{q}\ln(2\sigma^2) + N\ln\left(\frac{q}{2\Gamma(1/q)}\right) \\
&= -\frac{1}{2\sigma^2} \sum_{n=1}^N |y(\mathbf{x}_n, \mathbf{w}) - t_n|^q - \frac{N}{q}\ln(2\sigma^2) + \text{const}
\end{aligned}
$$
となり、式 (4.69) が厳密に導出されました。
特に $\mathbf{w}$ の関数として見ると、対数尤度の最大化は $L_q$ 損失関数 $\sum_{n=1}^N |y(\mathbf{x}_n, \mathbf{w}) - t_n|^q$ の最小化と厳密に等価です。 $\blacksquare$


In [10]:
# Exercise 4.11 の数値検証
# 規格化の検証
for test_q in [0.5, 1.0, 2.0, 3.0]:
    val_int, _ = integrate.quad(lambda x: generalized_gaussian_pdf(x, sigma2=1.2, q=test_q), -np.inf, np.inf)
    assert np.isclose(val_int, 1.0, atol=1e-5), f"Failed for q={test_q}"

# q=2 でのガウス分布との厳密一致
x_range = np.linspace(-4, 4, 100)
p_gen_2 = generalized_gaussian_pdf(x_range, sigma2=1.5, q=2.0)
p_norm = (1.0 / np.sqrt(2 * np.pi * 1.5)) * np.exp(- x_range**2 / (2 * 1.5))
assert np.allclose(p_gen_2, p_norm, atol=1e-14)

print("Exercise 4.11: 規格化・ガウス一致検証成功！")


Exercise 4.11: 規格化・ガウス一致検証成功！


---
## Exercise 4.12 (? ?): $L_q$ 損失の期待値最小化と条件付き中央値・最頻値

### 問題文
Consider the expected loss for regression problems under the $L_q$ loss function given by (4.40). Write down the condition that $y(\mathbf{x})$ must satisfy to minimize $\mathbb{E}[L_q]$. Show that, for $q = 1$, this solution represents the conditional median, i.e., the function $y(\mathbf{x})$ such that the probability mass for $t < y(\mathbf{x})$ is the same as for $t \ge y(\mathbf{x})$. Also show that the minimum expected $L_q$ loss for $q \to 0$ is given by the conditional mode, i.e., by the function $y(\mathbf{x})$ being equal to the value of $t$ that maximizes $p(t|\mathbf{x})$ for each $\mathbf{x}$.

---

### 数学的証明
#### 1. 一般の $q$ に対する停留条件
$L_q$ 損失関数の期待値は：
$$
\mathbb{E}[L_q] = \iint |y(\mathbf{x}) - t|^q p(\mathbf{x}, t) \, d\mathbf{x} \, dt = \int \left\{ \int |y(\mathbf{x}) - t|^q p(t|\mathbf{x}) \, dt \right\} p(\mathbf{x}) \, d\mathbf{x}
$$
各入力 $\mathbf{x}$ について括弧内の条件付き期待損失を最小化すれば十分です。
$y = y(\mathbf{x})$ に関して微分します（$|y - t|^q$ の微分は $q |y - t|^{q-1} \text{sgn}(y - t)$）：
$$
\frac{\partial}{\partial y} \int |y - t|^q p(t|\mathbf{x}) \, dt = q \int |y - t|^{q-1} \text{sgn}(y - t) \, p(t|\mathbf{x}) \, dt = 0
$$
したがって、$y(\mathbf{x})$ が満たすべき最適停留条件は：
$$
\int_{-\infty}^\infty |y(\mathbf{x}) - t|^{q-1} \text{sgn}(y(\mathbf{x}) - t) \, p(t|\mathbf{x}) \, dt = 0 \tag{Ans: 停留条件}
$$

#### 2. $q = 1$ の場合（条件付き中央値 / Conditional Median）
$q = 1$ のとき、$|y - t|^{1-1} = 1$ となるため、停留条件は：
$$
\int_{-\infty}^\infty \text{sgn}(y(\mathbf{x}) - t) \, p(t|\mathbf{x}) \, dt = 0
$$
積分領域を $t < y(\mathbf{x})$ と $t > y(\mathbf{x})$ に分割します：
$$
\int_{-\infty}^{y(\mathbf{x})} (+1) p(t|\mathbf{x}) \, dt + \int_{y(\mathbf{x})}^\infty (-1) p(t|\mathbf{x}) \, dt = 0
$$
$$
\implies \int_{-\infty}^{y(\mathbf{x})} p(t|\mathbf{x}) \, dt = \int_{y(\mathbf{x})}^\infty p(t|\mathbf{x}) \, dt
$$
確率の総和は $1$ であるため、両辺は $0.5$（$50\%$）に等しくなります：
$$
P(t < y(\mathbf{x})|\mathbf{x}) = P(t \ge y(\mathbf{x})|\mathbf{x}) = \frac{1}{2}
$$
これは、解 $y(\mathbf{x})$ が**条件付き中央値 (conditional median)** そのものであることを示しています。 $\blacksquare$

#### 3. $q \to 0$ の極限（条件付き最頻値 / Conditional Mode）
$q \to 0$ のとき、$|y - t|^q$ の振る舞いに着目します：
$$
\lim_{q \to 0^+} |y - t|^q = \begin{cases} 0 & (y = t) \\ 1 & (y \neq t) \end{cases}
$$
これはまさに $0-1$ 損失関数（誤りがあれば $1$、完全一致のみ $0$）です！
より厳密に、$q \to 0$ の極限での期待値を調べます：
$$
\int |y - t|^q p(t|\mathbf{x}) \, dt \approx \int_{|y - t| > \epsilon} 1 \cdot p(t|\mathbf{x}) \, dt = 1 - \int_{y - \epsilon}^{y + \epsilon} p(t|\mathbf{x}) \, dt \approx 1 - 2\epsilon \, p(y|\mathbf{x})
$$
したがって、この損失を最小化することは、局所確率密度 $p(y|\mathbf{x})$ を最大化することと厳密に等価です：
$$
\arg\min_y \lim_{q \to 0} \mathbb{E}[|y - t|^q|\mathbf{x}] = \arg\max_y p(y|\mathbf{x})
$$
これは**条件付き最頻値 (conditional mode)** に他なりません。 $\blacksquare$


In [11]:
# Exercise 4.12 の数値検証
# 非対称な混合ガウス分布
t_grid = np.linspace(-3, 5, 2500)
# 成分1: 平均 -1, 標準偏差 0.4, 重み 0.75
# 成分2: 平均 2.5, 標準偏差 0.8, 重み 0.25
pdf_asym = (
    0.75 * (1.0 / (0.4 * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((t_grid + 1.0) / 0.4)**2) +
    0.25 * (1.0 / (0.8 * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((t_grid - 2.5) / 0.8)**2)
)

# 1. 平均 (q=2)
mean_theoretical = 0.75 * (-1.0) + 0.25 * (2.5) # -0.125
y_q2 = compute_optimal_lq_predictor(None, q=2.0, t_grid=t_grid, pdf_values=pdf_asym)

# 2. 中央値 (q=1)
cdf_vals = np.cumsum(pdf_asym) * (t_grid[1] - t_grid[0])
median_theoretical = t_grid[np.searchsorted(cdf_vals, 0.5)]
y_q1 = compute_optimal_lq_predictor(None, q=1.0, t_grid=t_grid, pdf_values=pdf_asym)

# 3. 最頻値 (q -> 0, q=0.08)
mode_theoretical = t_grid[np.argmax(pdf_asym)]
y_q0 = compute_optimal_lq_predictor(None, q=0.08, t_grid=t_grid, pdf_values=pdf_asym)

print(f"q=2.0 解: {y_q2:.4f} (理論平均:   {mean_theoretical:.4f})")
print(f"q=1.0 解: {y_q1:.4f} (理論中央値: {median_theoretical:.4f})")
print(f"q=0.08解: {y_q0:.4f} (理論最頻値: {mode_theoretical:.4f})")

assert np.isclose(y_q2, mean_theoretical, atol=0.02)
assert np.isclose(y_q1, median_theoretical, atol=0.03)
assert np.isclose(y_q0, mode_theoretical, atol=0.05)
print("Exercise 4.12: 検証成功！")


q=2.0 解: -0.1262 (理論平均:   -0.1250)
q=1.0 解: -0.8265 (理論中央値: -0.8263)
q=0.08解: -0.9992 (理論最頻値: -0.9992)
Exercise 4.12: 検証成功！


In [12]:
# =====================================================================
# 第4章 演習問題 (Exercises 4.1 〜 4.12) 総括自己採点アサーション
# =====================================================================
print("Running complete verification of Chapter 4 Exercises...")

# Ex 4.1 & 4.2
w_41, A_41, T_41 = solve_polynomial_normal_equations(x_sample, t_sample, M=2)
w_42, A_42 = solve_regularized_polynomial_equations(x_sample, t_sample, M=2, lam=1.0)
assert la.norm(w_42) < la.norm(w_41)

# Ex 4.3
u0_t, u_t, _, _ = sigmoid_to_tanh_params(1.0, np.array([2.0, -1.0]), np.array([0.0, 1.0]), 0.5)
assert u0_t == 1.0 + 0.5 * (2.0 - 1.0)
assert np.allclose(u_t, np.array([1.0, -0.5]))

# Ex 4.4
P_test = compute_projection_matrix(Phi_mat)
assert np.allclose(P_test @ P_test, P_test)

# Ex 4.5
w_wls_t = solve_weighted_least_squares(Phi_wls, t_wls, weights_int)
assert np.allclose(w_wls_t, w_rep_sol)

# Ex 4.7
W_m_t, Sig_m_t = estimate_multivariate_regression_mle(Phi_multi, T_multi)
assert W_m_t.shape == (4, 3)

# Ex 4.11
assert np.isclose(integrate.quad(lambda x: generalized_gaussian_pdf(x, 1.0, 2.0), -np.inf, np.inf)[0], 1.0)

# Ex 4.12
assert np.isclose(y_q2, mean_theoretical, atol=0.05)

print("=" * 65)
print("ALL 12 EXERCISES IN CHAPTER 4 FULLY VERIFIED AND PASSED (100%)!")
print("=" * 65)


Running complete verification of Chapter 4 Exercises...
ALL 12 EXERCISES IN CHAPTER 4 FULLY VERIFIED AND PASSED (100%)!


---
## 第4章 演習問題の総括と第5章への接続

本演習問題ノートブックを通じて、第4章で学んだ単層ネットワーク（線形回帰モデル）の全理論体系を完璧に定着させることができました：

1. **正規方程式と正則化**: 多項式モデルにおける二乗和誤差および正則化二乗和誤差の最適解が、閉形式の連立線形方程式として表現されること（Ex 4.1, 4.2, 4.6）。
2. **基底関数の数学的柔軟性**: ロジスティック・シグモイド基底とハイパボリック・タンジェント基底の線形結合が、パラメータの単純な一次変換により厳密に等価であること（Ex 4.3）。
3. **直交射影幾何学の美しさ**: 最小二乗解 $\mathbf{y} = \boldsymbol{\Phi}\mathbf{w}_{\mathrm{ML}}$ が、高次元標本空間におけるターゲットベクトル $\mathbf{t}$ の列空間 $\mathcal{S}$ への直交射影であり、残差ベクトルが基底空間と厳密に直交すること（Ex 4.4）。
4. **重み付き回帰の統計的本質**: 不均一分散ノイズの精度重み付けと、観測標本の重複度が本質的に同一の数理構造を持つこと（Ex 4.5）。
5. **多変量回帰の分離定理**: 任意の共分散行列 $\boldsymbol{\Sigma}$ を持つ多変量正規ノイズに対しても、最尤推定解 $\mathbf{W}_{\mathrm{ML}}$ は各出力ごとに独立な最小二乗解と一致すること（Ex 4.7）。
6. **決定理論の統一的理解**: 変分法および二乗展開により、多変量二乗損失の最適予測が条件付き期待値 $\mathbb{E}[\mathbf{t}|\mathbf{x}]$ となる直交分解（Ex 4.8 - 4.10）。
7. **$L_q$ 損失と一般化ガウスノイズ**: ノイズ分布と損失関数の対応関係、および $q=2$（条件付き平均）、$q=1$（条件付き中央値）、$q \to 0$（条件付き最頻値）という最適予測の多様性（Ex 4.11, 4.12）。

これにより、**第4章（単層ネットワーク: 回帰）の全コンテンツ（4.1, 4.2, 4.3, 演習問題 4.1〜4.12）が 100% 完了**しました！
次章では、回帰から分類への跳躍となる**第5章 単層ネットワーク: 分類 (Single-layer Networks: Classification)** へと進みます！
